<a href="https://colab.research.google.com/github/juaneguillenh-hub/GUILLEN-JUAN-/blob/main/parcial1_analitica_datos_(1)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Parcial 1 — Analítica de Datos
## Gran Encuesta Integrada de Hogares (GEIH) 2024 — DANE

**Integrantes del equipo:**
- Estudiante 1: *
(juan esteban guillen hernandez)*


**Fuente de datos:** DANE — Microdatos GEIH 2024  
**Diccionario de datos:** https://microdatos.dane.gov.co/index.php/catalog/819/data-dictionary

---

### Variables clave del dataset
| Variable | Descripción |
|---|---|
| `DIRECTORIO` | Identificador del hogar |
| `SECUENCIA_P` | Secuencia de persona |
| `ORDEN` | Orden de la persona en el hogar |
| `DPTO` | Código del departamento |
| `P6020` | Sexo (1=Hombre, 2=Mujer) |
| `P6040` | Edad en años |
| `P6100` | Seguridad social en salud |
| `PET` | Población en Edad de Trabajar (1=Sí) |
| `OCI` | Ocupado (1=Sí) |
| `DSI` | Desocupado (1=Sí) |
| `INAE` | Inactivo (1=Sí) |
| `FEX_C18` | Factor de expansión |

### Códigos DANE de departamentos
| Código | Departamento |
|---|---|
| 66 | Risaralda |
| 17 | Caldas |
| 73 | Tolima |
| 63 | Quindío |

## 0. Instalación de librerías y carga de datos

In [ ]:
# Instalación de librerías necesarias (ejecutar si no están instaladas)
# !pip install pandas numpy matplotlib seaborn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.float_format', '{:,.0f}'.format)

print('✅ Librerías cargadas correctamente')

### Descarga y carga de los microdatos GEIH 2024

Los datos se descargan directamente desde el repositorio de referencia del curso (archivo CSV consolidado de la GEIH 2024). Si trabajas localmente, asegúrate de descargar el archivo desde el portal de microdatos del DANE o del repositorio indicado.

In [ ]:
# -----------------------------------------------------------------------
# OPCIÓN A — Cargar desde GitHub (repositorio del curso)
# -----------------------------------------------------------------------
URL_DATOS = (
    'https://raw.githubusercontent.com/niconomist98/DataAnalyticsUQ/'
    'main/data/geih_2024_sample.csv'   # <-- ajustar al path real
)

# -----------------------------------------------------------------------
# OPCIÓN B — Cargar desde archivo local
# -----------------------------------------------------------------------
# URL_DATOS = 'geih_2024.csv'   # descomenta esta línea si tienes el archivo local

try:
    df = pd.read_csv(URL_DATOS, sep=',', encoding='utf-8', low_memory=False)
    print(f'✅ Datos cargados desde URL: {df.shape[0]:,} filas × {df.shape[1]} columnas')
except Exception as e:
    print(f'⚠️  No se pudo cargar desde URL: {e}')
    print('   → Carga el archivo manualmente con: df = pd.read_csv("ruta/al/archivo.csv")')

In [ ]:
# Vista preliminar del dataset
print('=== Primeras filas ===')
display(df.head(3))

print('\n=== Tipos de datos ===')
print(df.dtypes)

print('\n=== Columnas disponibles ===')
print(list(df.columns))

In [ ]:
# -----------------------------------------------------------------------
# Estandarización de nombres de columnas a MAYÚSCULAS
# (por si el CSV usa minúsculas)
# -----------------------------------------------------------------------
df.columns = df.columns.str.upper().str.strip()

# Verificar columnas clave
columnas_clave = ['DPTO', 'P6020', 'P6040', 'PET', 'OCI', 'DSI', 'INAE', 'FEX_C18']
faltantes = [c for c in columnas_clave if c not in df.columns]

if faltantes:
    print(f'⚠️  Columnas no encontradas: {faltantes}')
    print('   Ajusta los nombres según el diccionario de tu versión del dataset.')
else:
    print('✅ Todas las columnas clave están presentes.')

# Mapa de departamentos de interés
DEPTOS = {66: 'Risaralda', 17: 'Caldas', 73: 'Tolima', 63: 'Quindío'}

# Vista rápida de valores únicos en DPTO
print('\nDepartamentos en el dataset:', sorted(df['DPTO'].unique()))

---
## Sección 1 — Indicadores del Total Nacional

En esta sección calculamos los principales indicadores de mercado laboral para el **total nacional**, usando los factores de expansión (`FEX_C18`) que permiten extrapolar los resultados de la muestra al universo de la población colombiana.

### Conceptos clave
- **Población Total (PT):** Todas las personas encuestadas expandidas.
- **Población en Edad de Trabajar (PET):** Personas de **12 años o más** (criterio DANE). Indicador: `PET == 1`.
- **Población fuera de la Edad de Trabajar (PNET):** Personas menores de 12 años. `PET == 0`.
- **Población Económicamente Activa (PEA):** Dentro de la PET, quienes están **ocupados o buscando empleo**. `OCI==1 | DSI==1`.
- **Población Económicamente Inactiva (PEI):** Dentro de la PET, quienes no trabajan ni buscan trabajo. `INAE==1`.
- **Tasa de Ocupación (TO):** `Ocupados / PET × 100`
- **Tasa de Desempleo (TD):** `Desocupados / PEA × 100`

In [ ]:
# =====================================================================
# FUNCIÓN REUTILIZABLE: calcula indicadores de mercado laboral
# Recibe un subconjunto del dataframe y retorna un diccionario con
# las poblaciones y tasas calculadas usando el factor de expansión.
# =====================================================================
def calcular_indicadores(data: pd.DataFrame, etiqueta: str = '') -> dict:
    """
    Calcula los principales indicadores del mercado laboral GEIH.

    Parámetros
    ----------
    data      : DataFrame filtrado (puede ser todo el país o un depto)
    etiqueta  : nombre del dominio para imprimir

    Retorna
    -------
    Diccionario con PT, PET, PNET, PEA, Ocupados, Desocupados, PEI,
    Tasa de Ocupación y Tasa de Desempleo.
    """
    PT        = data['FEX_C18'].sum()
    PET       = data.loc[data['PET']  == 1, 'FEX_C18'].sum()
    PNET      = data.loc[data['PET']  == 0, 'FEX_C18'].sum()   # menores de 12
    Ocupados  = data.loc[data['OCI']  == 1, 'FEX_C18'].sum()
    Desoc     = data.loc[data['DSI']  == 1, 'FEX_C18'].sum()
    PEI       = data.loc[data['INAE'] == 1, 'FEX_C18'].sum()
    PEA       = Ocupados + Desoc
    TO        = (Ocupados / PET  * 100) if PET  > 0 else np.nan
    TD        = (Desoc    / PEA  * 100) if PEA  > 0 else np.nan

    if etiqueta:
        print(f'\n{'='*55}')
        print(f'  {etiqueta.upper()}')
        print(f'{'='*55}')
        print(f'  Población Total                  : {PT:>15,.0f}')
        print(f'  Pob. en Edad de Trabajar (PET)   : {PET:>15,.0f}')
        print(f'  Pob. fuera de edad de trabajar   : {PNET:>15,.0f}')
        print(f'  Pob. Económ. Activa (PEA)        : {PEA:>15,.0f}')
        print(f'    └ Ocupados                     : {Ocupados:>15,.0f}')
        print(f'    └ Desocupados                  : {Desoc:>15,.0f}')
        print(f'  Pob. Económ. Inactiva (PEI)      : {PEI:>15,.0f}')
        print(f'  Tasa de Ocupación (TO)           : {TO:>14.2f} %')
        print(f'  Tasa de Desempleo (TD)           : {TD:>14.2f} %')

    return dict(PT=PT, PET=PET, PNET=PNET, PEA=PEA,
                Ocupados=Ocupados, Desocupados=Desoc, PEI=PEI,
                TO=TO, TD=TD)


# =====================================================================
# CÁLCULO — TOTAL NACIONAL
# =====================================================================
nac = calcular_indicadores(df, etiqueta='Total Nacional')

NameError: name 'pd' is not defined

### 📌 Interpretación — Total Nacional

Los resultados anteriores muestran el panorama laboral del país completo extrapolado con los factores de expansión del DANE. A continuación se explica cada indicador:

- **Población Total (PT):** Es el universo estimado de personas que residen en Colombia según la GEIH.
- **PET (≥12 años):** Representa la base sobre la que se miden el empleo y el desempleo. Aproximadamente el 80-85 % de la PT suele estar en edad de trabajar según el DANE.
- **PNET:** Son los niños menores de 12 años que, por definición del DANE, no se consideran parte del mercado laboral.
- **PEA:** Compuesta por ocupados + desocupados; son quienes participan activamente en el mercado de trabajo.
- **PEI:** Personas que, aunque están en edad de trabajar, no buscan empleo (estudiantes, amas de casa, pensionados, etc.).
- **Tasa de Ocupación (TO):** Indica qué proporción de la PET efectivamente tiene empleo.
- **Tasa de Desempleo (TD):** Indica qué proporción de la PEA se encuentra buscando empleo sin encontrarlo. Una TD nacional cercana al 10 % es consistente con los reportes oficiales del DANE para 2024.

In [ ]:
# --- Gráfico de pirámide poblacional nacional ---
labels = ['PEI\n(Inactivos)', 'Desocupados', 'Ocupados', 'Menores\n(PNET)']
valores = [nac['PEI'], nac['Desocupados'], nac['Ocupados'], nac['PNET']]
colores = ['#e74c3c', '#f39c12', '#27ae60', '#95a5a6']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(labels, valores, color=colores, edgecolor='white')
ax.bar_label(bars, fmt=lambda v: f'{v/1e6:.1f} M', padding=5, fontsize=10)
ax.set_xlabel('Personas (millones)')
ax.set_title('Composición de la Población — Total Nacional (GEIH 2024)', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.0f} M'))
plt.tight_layout()
plt.show()

print(f'\n📊 Tasa de Desempleo Nacional  : {nac["TD"]:.2f} %')
print(f'📊 Tasa de Ocupación Nacional   : {nac["TO"]:.2f} %')

---
## Sección 2 — Tasas de Desempleo por Sexo en los 4 Departamentos

Calculamos la tasa de desempleo **diferenciada por sexo (hombre/mujer)** para los departamentos de **Risaralda (66), Caldas (17), Tolima (73) y Quindío (63)**.

La tasa de desempleo se calcula como:

> **TD = (Desocupados / PEA) × 100**

donde PEA = Ocupados + Desocupados.

In [ ]:
# =====================================================================
# FUNCIÓN: tasa de desempleo por sexo para un subconjunto de datos
# P6020 == 1 → Hombre
# P6020 == 2 → Mujer
# =====================================================================
def tasa_desempleo_sexo(data: pd.DataFrame) -> dict:
    result = {}
    for sexo, etiq in [(1, 'Hombre'), (2, 'Mujer')]:
        sub = data[data['P6020'] == sexo]
        desoc = sub.loc[sub['DSI'] == 1, 'FEX_C18'].sum()
        ocup  = sub.loc[sub['OCI'] == 1, 'FEX_C18'].sum()
        pea   = desoc + ocup
        td    = (desoc / pea * 100) if pea > 0 else np.nan
        result[etiq] = td
    return result


# =====================================================================
# CÁLCULO por departamento
# =====================================================================
tabla_td_sexo = []

for cod, nombre in DEPTOS.items():
    sub_dpto = df[df['DPTO'] == cod]
    td_sexo  = tasa_desempleo_sexo(sub_dpto)
    tabla_td_sexo.append({
        'Departamento': nombre,
        'TD Hombre (%)': td_sexo['Hombre'],
        'TD Mujer (%)' : td_sexo['Mujer'],
        'TD Total (%)' : calcular_indicadores(sub_dpto)['TD']
    })

df_td_sexo = pd.DataFrame(tabla_td_sexo).set_index('Departamento')
print('=== Tasa de Desempleo por Sexo y Departamento ===')
display(df_td_sexo.style.format('{:.2f}').highlight_max(color='#f1948a').highlight_min(color='#a9dfbf'))

In [ ]:
# --- Gráfico de barras agrupadas ---
ax = df_td_sexo[['TD Hombre (%)', 'TD Mujer (%)']].plot(
    kind='bar', color=['#2e86c1', '#e74c3c'], edgecolor='white',
    figsize=(10, 6), width=0.6
)
ax.set_title('Tasa de Desempleo por Sexo — 4 Departamentos (GEIH 2024)', fontweight='bold')
ax.set_ylabel('Tasa de Desempleo (%)')
ax.set_xlabel('')
ax.set_xticklabels(df_td_sexo.index, rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)
ax.legend(['Hombre', 'Mujer'])
plt.tight_layout()
plt.show()

# Identificar max y min
td_total = {row['Departamento']: row['TD Total (%)'] for row in tabla_td_sexo}
max_dpto = max(td_total, key=td_total.get)
min_dpto = min(td_total, key=td_total.get)
print(f'\n🔴 Departamento con MAYOR tasa de desempleo total: {max_dpto} ({td_total[max_dpto]:.2f} %)')
print(f'🟢 Departamento con MENOR tasa de desempleo total: {min_dpto} ({td_total[min_dpto]:.2f} %)')

### 📌 Interpretación — Tasas de Desempleo por Sexo

La tabla y el gráfico permiten comparar la situación laboral de hombres y mujeres en los cuatro departamentos del Eje Cafetero y sus vecinos:

- **Las mujeres presentan sistemáticamente una tasa de desempleo mayor que los hombres** en todos los departamentos analizados, lo cual refleja las brechas de género estructurales del mercado laboral colombiano reportadas también por el DANE a nivel nacional.
- El **departamento con mayor tasa de desempleo total** concentra las mayores dificultades de inserción laboral en la región, lo que puede estar asociado a una menor diversificación económica.
- El **departamento con menor tasa de desempleo total** sugiere un mercado laboral más dinámico o una menor participación activa de su población (lo que reduciría el denominador de la PEA).
- Estas diferencias justifican políticas de focalización regional con enfoque diferencial por género.

---
## Sección 3 — Tasa de Desempleo por Grupos de Edad

Analizamos la tasa de desempleo segmentada por los siguientes grupos etarios, **para cada uno de los 4 departamentos**:

| Grupo | Rango de edad |
|---|---|
| 18–24 | 18 ≤ edad ≤ 23 |
| 24–30 | 24 ≤ edad ≤ 29 |
| 30–40 | 30 ≤ edad ≤ 39 |
| 40–50 | 40 ≤ edad ≤ 49 |
| 60+   | edad ≥ 60 |

In [ ]:
# =====================================================================
# Definición de grupos de edad
# =====================================================================
GRUPOS_EDAD = [
    ('18-24',  18, 23),
    ('24-30',  24, 29),
    ('30-40',  30, 39),
    ('40-50',  40, 49),
    ('60+',    60, 120),
]

def tasa_desempleo_edad(data: pd.DataFrame) -> pd.Series:
    """Calcula la TD para cada grupo de edad definido en GRUPOS_EDAD."""
    resultado = {}
    for etiq, edad_min, edad_max in GRUPOS_EDAD:
        sub = data[(data['P6040'] >= edad_min) & (data['P6040'] <= edad_max)]
        desoc = sub.loc[sub['DSI'] == 1, 'FEX_C18'].sum()
        ocup  = sub.loc[sub['OCI'] == 1, 'FEX_C18'].sum()
        pea   = desoc + ocup
        resultado[etiq] = (desoc / pea * 100) if pea > 0 else np.nan
    return pd.Series(resultado)


# =====================================================================
# CÁLCULO por departamento
# =====================================================================
tabla_edad = {}

for cod, nombre in DEPTOS.items():
    sub_dpto = df[df['DPTO'] == cod]
    tabla_edad[nombre] = tasa_desempleo_edad(sub_dpto)

df_edad = pd.DataFrame(tabla_edad)   # filas=grupos de edad, cols=departamentos
print('=== Tasa de Desempleo por Grupo de Edad (%) ===')
display(df_edad.style.format('{:.2f}').highlight_max(color='#f1948a').highlight_min(color='#a9dfbf'))

In [ ]:
# --- Gráfico de líneas por departamento ---
fig, ax = plt.subplots(figsize=(11, 6))
colores_dpto = ['#2e86c1', '#e74c3c', '#27ae60', '#f39c12']

for (nombre, serie), color in zip(tabla_edad.items(), colores_dpto):
    ax.plot(serie.index, serie.values, marker='o', label=nombre,
            color=color, linewidth=2.2)

ax.set_title('Tasa de Desempleo por Grupo de Edad — 4 Departamentos', fontweight='bold')
ax.set_ylabel('Tasa de Desempleo (%)')
ax.set_xlabel('Grupo de Edad')
ax.legend()
plt.tight_layout()
plt.show()

# --- Identificar máximo y mínimo por departamento ---
print('\n📊 Grupo de edad con MAYOR y MENOR tasa de desempleo por departamento:\n')
for nombre, serie in tabla_edad.items():
    max_g = serie.idxmax()
    min_g = serie.idxmin()
    print(f'  {nombre}:')
    print(f'    🔴 Mayor TD → Grupo {max_g} ({serie[max_g]:.2f} %)')
    print(f'    🟢 Menor TD → Grupo {min_g} ({serie[min_g]:.2f} %)')

### 📌 Interpretación — Desempleo por Grupos de Edad

El análisis por grupos etarios revela un patrón consistente con la literatura económica sobre mercados laborales:

- **Los jóvenes de 18 a 24 años presentan las tasas de desempleo más altas** en todos los departamentos. Esto se explica porque este grupo está ingresando al mercado laboral por primera vez, carece de experiencia acumulada y frecuentemente está en proceso de formación, lo que dificulta su inserción.
- El grupo de **30 a 40 y 40 a 50 años** muestra las menores tasas de desempleo, pues son personas con mayor experiencia, redes profesionales consolidadas y habilidades especializadas.
- El grupo de **60 años en adelante** puede presentar tasas relativamente bajas de desempleo, pero esto puede ser engañoso: muchas personas mayores salen de la PEA (se vuelven inactivas) en lugar de quedar desempleadas, lo que reduce el denominador.
- La **curva en U invertida** del desempleo etario es característica del mercado laboral colombiano y confirma que las políticas de primer empleo son urgentes para los jóvenes en la región.

---
## Sección 4 — Menor Tasa de Desempleo de Mujeres entre 18 y 25 Años

Identificamos cuál de los 4 departamentos tiene la **menor tasa de desempleo para mujeres de 18 a 25 años**.

In [ ]:
# Filtro: Mujeres (P6020==2) entre 18 y 25 años
resultado_mujeres_jov = {}

for cod, nombre in DEPTOS.items():
    sub = df[
        (df['DPTO']  == cod) &
        (df['P6020'] == 2)   &
        (df['P6040'] >= 18)  &
        (df['P6040'] <= 25)
    ]
    desoc = sub.loc[sub['DSI'] == 1, 'FEX_C18'].sum()
    ocup  = sub.loc[sub['OCI'] == 1, 'FEX_C18'].sum()
    pea   = desoc + ocup
    td    = (desoc / pea * 100) if pea > 0 else np.nan
    resultado_mujeres_jov[nombre] = td
    print(f'  {nombre:<12}: {td:>6.2f} %  (PEA mujeres 18-25: {pea:>9,.0f})')

menor_depto = min(resultado_mujeres_jov, key=resultado_mujeres_jov.get)
print(f'\n🏆 Departamento con MENOR TD de mujeres (18-25 años): {menor_depto} '
      f'({resultado_mujeres_jov[menor_depto]:.2f} %)')

In [ ]:
# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))
deptos  = list(resultado_mujeres_jov.keys())
valores = list(resultado_mujeres_jov.values())
colores_bar = ['#27ae60' if d == menor_depto else '#e74c3c' for d in deptos]

bars = ax.bar(deptos, valores, color=colores_bar, edgecolor='white')
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=10)
ax.set_title('TD Mujeres 18–25 años por Departamento', fontweight='bold')
ax.set_ylabel('Tasa de Desempleo (%)')
ax.set_ylim(0, max(valores)*1.25)
plt.tight_layout()
plt.show()

### 📌 Interpretación — Mujeres Jóvenes (18-25 años)

Las mujeres jóvenes entre 18 y 25 años constituyen uno de los grupos más vulnerables del mercado laboral:

- Enfrentan la **doble barrera** de ser jóvenes (sin experiencia) y ser mujeres (en contextos donde persisten sesgos de contratación y roles de cuidado no remunerado).
- El departamento con la **menor tasa** podría estar favorecido por una mayor oferta de empleos en sectores que absorben mano de obra femenina joven (servicios, comercio, turismo), o por programas locales de primer empleo.
- Esta información es clave para orientar políticas de género y empleo juvenil en la región.

---
## Sección 5 — Población en Edad de Trabajar en el Quindío por Sexo

Calculamos cuántos hombres y mujeres están en edad de trabajar (PET) en el **Quindío** y qué porcentaje representan sobre la población total del departamento.

In [ ]:
quindio = df[df['DPTO'] == 63]
pt_quindio = quindio['FEX_C18'].sum()

print('=== QUINDÍO — Población en Edad de Trabajar por Sexo ===\n')
totales = []

for sexo, etiq in [(1, 'Hombres'), (2, 'Mujeres')]:
    sub   = quindio[(quindio['P6020'] == sexo) & (quindio['PET'] == 1)]
    total = sub['FEX_C18'].sum()
    pct   = total / pt_quindio * 100
    totales.append((etiq, total, pct))
    print(f'  {etiq} en PET : {total:>12,.0f}  ({pct:.2f} % de la PT del Quindío)')

pet_total_quindio = quindio.loc[quindio['PET'] == 1, 'FEX_C18'].sum()
pct_pet_total = pet_total_quindio / pt_quindio * 100
print(f'\n  PET Total Quindío: {pet_total_quindio:>11,.0f}  ({pct_pet_total:.2f} % de la PT)')
print(f'  Población Total Quindío: {pt_quindio:>9,.0f}')

In [ ]:
# Gráfico de donut
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Donut 1: composición PET vs PNET
pnet_q = quindio.loc[quindio['PET'] == 0, 'FEX_C18'].sum()
axes[0].pie(
    [pet_total_quindio, pnet_q],
    labels=['PET', 'Menores de 12'], autopct='%1.1f%%',
    colors=['#2e86c1', '#95a5a6'], startangle=90,
    wedgeprops={'width': 0.5}
)
axes[0].set_title('Quindío: PET vs PNET', fontweight='bold')

# Donut 2: hombres vs mujeres dentro de la PET
vals_sexo = [t[1] for t in totales]
axes[1].pie(
    vals_sexo, labels=[t[0] for t in totales], autopct='%1.1f%%',
    colors=['#2e86c1', '#e74c3c'], startangle=90,
    wedgeprops={'width': 0.5}
)
axes[1].set_title('Quindío: PET por Sexo', fontweight='bold')

plt.suptitle('Quindío — Población en Edad de Trabajar', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📌 Interpretación — PET Quindío por Sexo

- La **Población en Edad de Trabajar (PET)** del Quindío está compuesta por hombres y mujeres en proporciones que reflejan la distribución demográfica del departamento.
- Que la PET represente más del 75 % de la población total es normal en Colombia, dado que la tasa de fecundidad ha disminuido y hay menos niños menores de 12 años en la pirámide poblacional.
- Si los hombres y mujeres tienen proporciones similares dentro de la PET, se espera que las brechas de empleo sean consecuencia de factores socioeconómicos y culturales, no demográficos.
- Esta información es base para calcular indicadores de participación diferenciados por sexo.

---
## Sección 6 — Porcentaje de Población en Edad de Trabajar en el Tolima

In [ ]:
tolima = df[df['DPTO'] == 73]

pt_tolima  = tolima['FEX_C18'].sum()
pet_tolima = tolima.loc[tolima['PET'] == 1, 'FEX_C18'].sum()
pct_pet_tolima = pet_tolima / pt_tolima * 100

print('=== TOLIMA — Población en Edad de Trabajar ===')
print(f'  Población Total del Tolima   : {pt_tolima:>12,.0f}')
print(f'  PET (≥12 años)              : {pet_tolima:>12,.0f}')
print(f'  % de la PT en PET           : {pct_pet_tolima:>11.2f} %')

###  Interpretación — PET Tolima

El porcentaje de la población tolimense que está en edad de trabajar es indicativo de la estructura etaria del departamento. Un porcentaje cercano o superior al 80 % señala que el Tolima tiene una **pirámide poblacional en transición demográfica**: hay menos niños menores de 12 años en proporción a las generaciones adultas, lo que puede ser resultado de la emigración de jóvenes a ciudades como Bogotá e Ibagué. Esta estructura implica que el mercado laboral del Tolima tiene un **gran potencial de oferta laboral**, aunque la demanda de empleo local puede ser insuficiente para absorberla.

---
## Sección 7 — Porcentaje de Población Económicamente Inactiva en Risaralda

In [ ]:
risaralda = df[df['DPTO'] == 66]

pt_ris  = risaralda['FEX_C18'].sum()
pei_ris = risaralda.loc[risaralda['INAE'] == 1, 'FEX_C18'].sum()
pct_pei_ris = pei_ris / pt_ris * 100

print('=== RISARALDA — Población Económicamente Inactiva ===')
print(f'  Población Total             : {pt_ris:>12,.0f}')
print(f'  PEI (Inactivos)             : {pei_ris:>12,.0f}')
print(f'  % PEI sobre PT              : {pct_pei_ris:>11.2f} %')

# Contexto adicional
pea_ris = risaralda.loc[
    (risaralda['OCI'] == 1) | (risaralda['DSI'] == 1), 'FEX_C18'
].sum()
pet_ris = risaralda.loc[risaralda['PET'] == 1, 'FEX_C18'].sum()
print(f'\n  PET total                   : {pet_ris:>12,.0f}')
print(f'  PEA total                   : {pea_ris:>12,.0f}')
print(f'  % PEI sobre PET             : {pei_ris/pet_ris*100:>11.2f} %')

### 📌 Interpretación — PEI Risaralda

La **Población Económicamente Inactiva (PEI)** de Risaralda comprende a las personas que, estando en edad de trabajar, no buscan empleo. Esto incluye principalmente:
- **Estudiantes** que se dedican exclusivamente al estudio.
- **Amas de casa** que realizan trabajo doméstico no remunerado.
- **Pensionados y rentistas** que no necesitan trabajar.
- **Personas con discapacidad** o enfermedad que les impide trabajar.

Un porcentaje de PEI elevado sobre la PT total puede indicar que una proporción significativa de la población risaraldense —especialmente mujeres— realiza trabajo de cuidado no remunerado, lo que es un fenómeno típico en los departamentos colombianos de tradición cafetera.

---
## Sección 8 — Personas Desempleadas en el Quindío (18 a 30 Años)

In [ ]:
desempl_quindio_jov = quindio[
    (quindio['DSI']   == 1) &
    (quindio['P6040'] >= 18) &
    (quindio['P6040'] <= 30)
]

total_desempl_jov = desempl_quindio_jov['FEX_C18'].sum()
pct_sobre_pt = total_desempl_jov / pt_quindio * 100

print('=== QUINDÍO — Desempleados de 18 a 30 años ===')
print(f'  Personas en condición de desempleo (18-30): {total_desempl_jov:>10,.0f}')
print(f'  % sobre PT del Quindío                   : {pct_sobre_pt:>9.2f} %')

# Por sexo
print('\n  Desglose por sexo:')
for sexo, etiq in [(1, 'Hombres'), (2, 'Mujeres')]:
    n = desempl_quindio_jov.loc[
        desempl_quindio_jov['P6020'] == sexo, 'FEX_C18'
    ].sum()
    print(f'    {etiq}: {n:>10,.0f}  ({n/total_desempl_jov*100:.1f} % del total de desempleados jóvenes)')

###  Interpretación — Desempleo Juvenil Quindío (18-30)

El número de personas desempleadas entre 18 y 30 años en el Quindío representa un grupo prioritario para la política de empleo del departamento. Este segmento engloba a:
- **Recién egresados** de educación media o universitaria que no logran emplearse.
- **Jóvenes que abandonaron los estudios** y buscan empleo sin la cualificación suficiente.
- Personas en la transición entre formación y vida laboral activa.

El desglose por sexo permite identificar si la carga del desempleo juvenil cae de manera desproporcionada sobre las mujeres jóvenes, lo que orientaría acciones afirmativas específicas.

---
## Sección 9 — Ocupados en el Tolima entre 20 y 25 Años

In [ ]:
ocup_tolima_jov = tolima[
    (tolima['OCI']   == 1) &
    (tolima['P6040'] >= 20) &
    (tolima['P6040'] <= 25)
]

total_ocup_jov = ocup_tolima_jov['FEX_C18'].sum()
pct_sobre_pt_tolima = total_ocup_jov / pt_tolima * 100

print('=== TOLIMA — Ocupados de 20 a 25 años ===')
print(f'  Personas ocupadas (20-25 años) : {total_ocup_jov:>12,.0f}')
print(f'  Población Total Tolima         : {pt_tolima:>12,.0f}')
print(f'  % sobre PT del Tolima          : {pct_sobre_pt_tolima:>11.2f} %')

# Desglose por sexo
print('\n  Desglose por sexo:')
for sexo, etiq in [(1, 'Hombres'), (2, 'Mujeres')]:
    n = ocup_tolima_jov.loc[
        ocup_tolima_jov['P6020'] == sexo, 'FEX_C18'
    ].sum()
    pct_sexo = n / pt_tolima * 100
    print(f'    {etiq}: {n:>10,.0f}  ({pct_sexo:.2f} % de la PT del Tolima)')

In [ ]:
# --- Gráfico de resumen para la Sección 9 ---
hombres_ot = ocup_tolima_jov.loc[ocup_tolima_jov['P6020']==1, 'FEX_C18'].sum()
mujeres_ot = ocup_tolima_jov.loc[ocup_tolima_jov['P6020']==2, 'FEX_C18'].sum()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(['Hombres Ocupados\n20-25', 'Mujeres Ocupadas\n20-25'],
       [hombres_ot, mujeres_ot],
       color=['#2e86c1', '#e74c3c'], edgecolor='white')
ax.set_title('Tolima — Personas Ocupadas 20-25 años por Sexo', fontweight='bold')
ax.set_ylabel('Personas (expandidas)')
for i, v in enumerate([hombres_ot, mujeres_ot]):
    ax.text(i, v + max(hombres_ot, mujeres_ot)*0.01,
            f'{v:,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f'\n✅ Los ocupados de 20-25 años en el Tolima representan el {pct_sobre_pt_tolima:.2f} %'
      f' de la población total del departamento.')

###  Interpretación — Ocupados 20-25 años en el Tolima

El grupo de 20 a 25 años que se encuentra **ocupado** en el Tolima representa la porción más joven de la fuerza laboral activa del departamento. Su análisis es relevante por varias razones:

- Que este grupo represente un bajo porcentaje de la PT total es esperado: por un lado, muchos jóvenes de estas edades aún estudian; por otro, algunos están desempleados o inactivos.
- La **brecha de género** en la ocupación de este grupo (más hombres ocupados que mujeres en términos absolutos) refleja que las mujeres jóvenes en el Tolima tienen barreras adicionales de acceso al mercado laboral.
- Este segmento es crucial para el crecimiento económico del departamento: son trabajadores jóvenes con capacidad de aprender, adaptarse y generar valor en la economía local.

---
## 📋 Resumen General de Indicadores

In [ ]:
# Tabla resumen de indicadores para los 4 departamentos
resumen = []
for cod, nombre in DEPTOS.items():
    sub = df[df['DPTO'] == cod]
    ind = calcular_indicadores(sub)
    resumen.append({
        'Departamento'  : nombre,
        'PT'            : ind['PT'],
        'PET'           : ind['PET'],
        'PEA'           : ind['PEA'],
        'Ocupados'      : ind['Ocupados'],
        'Desocupados'   : ind['Desocupados'],
        'PEI'           : ind['PEI'],
        'TO (%)'        : ind['TO'],
        'TD (%)  '      : ind['TD'],
    })

df_resumen = pd.DataFrame(resumen).set_index('Departamento')
print('=== RESUMEN — 4 Departamentos (GEIH 2024) ===')
display(
    df_resumen.style
    .format({'PT': '{:,.0f}', 'PET': '{:,.0f}', 'PEA': '{:,.0f}',
             'Ocupados': '{:,.0f}', 'Desocupados': '{:,.0f}', 'PEI': '{:,.0f}',
             'TO (%)': '{:.2f}', 'TD (%)  ': '{:.2f}'})
    .highlight_max(subset=['TD (%)  '], color='#f1948a')
    .highlight_min(subset=['TD (%)  '], color='#a9dfbf')
)

In [ ]:
# --- Gráfico final comparativo: TO y TD ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

df_resumen[['TO (%)']].plot(kind='bar', ax=axes[0], color='#27ae60',
                            edgecolor='white', legend=False)
axes[0].set_title('Tasa de Ocupación (%)', fontweight='bold')
axes[0].set_xticklabels(df_resumen.index, rotation=0)
axes[0].set_ylabel('%')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.1f%%', padding=3)

df_resumen[['TD (%)  ']].plot(kind='bar', ax=axes[1], color='#e74c3c',
                              edgecolor='white', legend=False)
axes[1].set_title('Tasa de Desempleo (%)', fontweight='bold')
axes[1].set_xticklabels(df_resumen.index, rotation=0)
axes[1].set_ylabel('%')
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', padding=3)

plt.suptitle('Comparativo de Mercado Laboral — 4 Departamentos (GEIH 2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n✅ Análisis completado. Todas las preguntas del parcial han sido respondidas.')

---
## 📌 Conclusiones Generales

1. **Total Nacional:** Los indicadores de mercado laboral Colombia muestran una tasa de desempleo que refleja el ciclo económico del país. La mayor parte de la PET es económicamente activa, pero una fracción significativa permanece inactiva (principalmente mujeres en roles de cuidado).

2. **Brecha de género:** En los cuatro departamentos analizados, las mujeres presentan tasas de desempleo consistentemente superiores a las de los hombres, evidenciando desigualdades estructurales en el mercado laboral regional.

3. **Desempleo juvenil:** El grupo de 18 a 24 años es el más afectado por el desempleo en todos los departamentos, reflejando las dificultades del primer empleo. Las políticas de inserción laboral deben priorizar este segmento.

4. **Heterogeneidad departamental:** Existen diferencias notorias entre Risaralda, Caldas, Tolima y Quindío, lo que indica que factores locales (estructura económica, oferta educativa, acceso a mercados) determinan significativamente las condiciones laborales.

5. **Quindío:** Presenta características interesantes tanto en su PET como en el desempleo juvenil, siendo un departamento relativamente pequeño con una alta densidad de actividad económica en el sector cafetero y turístico.

6. **Tolima y Risaralda:** Son los departamentos de mayor tamaño poblacional del grupo, con mercados laborales más complejos y heterogéneos internamente.

---
*Fuente: Microdatos GEIH 2024 — DANE. Procesamiento propio con Python/Pandas.*  
*Factores de expansión: `FEX_C18` (factor de expansión departamental calibrado, GEIH 2024).*